# Data Quality - Smoke Test Notebook

Run this notebook before enabling schedules or after major changes.
It verifies required tables, columns, and core data integrity contracts.

In [ ]:
# Configuration
try:
    from data_quality_config import LAKEHOUSE_NAME, SCHEMA_NAME
except Exception:
    LAKEHOUSE_NAME = "MyLakehouse"
    SCHEMA_NAME = "data_quality"

FULL_SCHEMA = f"{LAKEHOUSE_NAME}.{SCHEMA_NAME}"
print(f"Smoke test schema: {FULL_SCHEMA}")

In [ ]:
from IPython.display import display

errors = []

required_tables = {"check_registry", "validation_results"}
tables_df = spark.sql(f"SHOW TABLES IN {FULL_SCHEMA}").toPandas()
display(tables_df)

name_col = None
if "tableName" in tables_df.columns:
    name_col = "tableName"
else:
    lowered = {str(c).strip().lower(): c for c in tables_df.columns}
    for candidate in ["tablename", "table_name", "name", "table"]:
        if candidate in lowered:
            name_col = lowered[candidate]
            break

if name_col is None:
    errors.append("SHOW TABLES output has no table name column")
    found_tables = set()
else:
    found_tables = set(tables_df[name_col].astype(str).str.strip().str.lower().tolist())

missing_tables = sorted(required_tables - found_tables)
if missing_tables:
    errors.append("Missing required tables: " + ", \
,
,
check_registry" in found_tables:
    check_cols_df = spark.sql(f"DESCRIBE {FULL_SCHEMA}.check_registry").toPandas()
    check_cols = set(check_cols_df["col_name"].astype(str).str.strip().str.lower().tolist())
    required_check_cols = {
        "check_name", "model_name", "workspace_id", "dataset_id",
        "workspace_name", "dataset_name", "dax_expression", "run_frequency", "is_active"
    }
    missing_check_cols = sorted(required_check_cols - check_cols)
    if missing_check_cols:
        errors.append("check_registry missing columns: " + ", \
,
,
validation_results" in found_tables:
    result_cols_df = spark.sql(f"DESCRIBE {FULL_SCHEMA}.validation_results").toPandas()
    result_cols = set(result_cols_df["col_name"].astype(str).str.strip().str.lower().tolist())
    required_result_cols = {
        "run_id", "run_date", "run_timestamp", "check_name", "model_name",
        "workspace_id", "dataset_id", "workspace_name", "dataset_name",
        "result_value", "baseline_model", "baseline_value",
        "absolute_delta", "delta_pct", "status", "error_message"
    }
    missing_result_cols = sorted(required_result_cols - result_cols)
    if missing_result_cols:
        errors.append("validation_results missing columns: " + ", \
,
,
check_registry" in found_tables:
    dupe_count = spark.sql(f"""
        SELECT COUNT(*) AS c
        FROM (
            SELECT check_name, workspace_id, dataset_id
            FROM {FULL_SCHEMA}.check_registry
            GROUP BY check_name, workspace_id, dataset_id
            HAVING COUNT(*) > 1
        ) d
    """).collect()[0]["c"]
    if dupe_count > 0:
        errors.append(f"Duplicate identity keys found in check_registry: {dupe_count}")

    missing_active_required = spark.sql(f"""
        SELECT COUNT(*) AS c
        FROM {FULL_SCHEMA}.check_registry
        WHERE is_active = true
          AND (
              workspace_id IS NULL OR TRIM(workspace_id) = ''
              OR dataset_id IS NULL OR TRIM(dataset_id) = ''
              OR workspace_name IS NULL OR TRIM(workspace_name) = ''
              OR dataset_name IS NULL OR TRIM(dataset_name) = ''
              OR UPPER(TRIM(COALESCE(run_frequency, ''))) NOT IN ('ONCE_PER_DAY', 'MULTIPLE_PER_DAY')
          )
    """).collect()[0]["c"]
    if missing_active_required > 0:
        errors.append(
            f"Active check_registry rows with invalid required fields/frequency: {missing_active_required}"
        )

if errors:
    print("\nSMOKE TEST FAILED:")
    for e in errors:
        print(f" - {e}")
    raise RuntimeError("Smoke test failed. Resolve issues before scheduling production jobs.")

print("\nSMOKE TEST PASSED: Core data quality contracts are healthy.")